# In Silico Optimization
While fast, gradient solvers have a fatal flaw in biology: they get stuck in local optima (fake, smaller peaks) and they do not understand uncertainty. To conquer complex bioprocess manifolds, the industry standard is Bayesian Optimization (BO)
## 🧭 The Bayesian Paradigm: Exploration vs. Exploitation
Bayesian Optimization does not just blindly step uphill. It acts like an intelligent scientist. It looks at the data, forms a hypothesis about where the peak might be, calculates how confident it is, and then deliberately chooses the next best experiment to run.\
It balances two competing desires:\
- Exploitation: Searching near known high-titer areas to push slightly higher.
- Exploration: Searching completely untested areas of the DoE space because the uncertainty is so high that a massive, undiscovered peak might exist there.
To do this, Bayesian Optimization relies on two mathematical pillars:

### Pillar 1: The Surrogate Model (Gaussian Processes)
Instead of fitting a rigid polynomial or a deterministic neural network, BO typically uses a Gaussian Process (GP) (also known as Kriging).\
A GP does not just predict the mean titer; it predicts a distribution of possible titers. For any given Temperature and pH, the GP outputs: "I predict the titer is 5.5 g/L, with a 95% confidence interval of $\pm$ 0.8 g/L.\
"Where you have dense data, the uncertainty is near zero. Where you have no data, the uncertainty explodes.
### Pillar 2: The Acquisition Function
This is the mathematical formula that tells the algorithm exactly which bioreactor condition to test next. The most common is Expected Improvement (EI).\
EI looks at the Gaussian Process and calculates a score for every possible combination of Temperature and pH. It rewards conditions that have a high predicted titer and conditions that have high uncertainty. The algorithm finds the highest EI score, and that becomes your next physical laboratory experiment.
## 💻 Exercise: Coding the Bayesian Optimizer
Use of the `scikit-optimize` (`skopt`) library. This is how to set up an automated "Active Learning" loop to find the peak of our `true_bioreactor` function with the absolute minimum number of experiments.\
**Task:** Read how we define the boundaries of the biological space and let the Gaussian Process hunt for the peak.

In [4]:
import numpy as np
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args

In [15]:
def true_bioreactor(X):
    Temp, pH, FS, Gl, NaBu = X[:, 0], X[:, 1], X[:, 2], X[:, 3],X[:, 4]

    # The true peak is hidden at: Temp=32.8, pH=6.95, Glucose=5.5
    # Feed Start (FS) and NaBu have tiny coefficients (noise)
    titer = (
    6.5 - 0.8*(Temp-36.8)**2 - 2.5*(pH-6.8)**2 - 0.4*(Gl-5.5)**2 + 0.6*(Temp-36.8)*(pH-6.8) + 0.01*FS - 0.03*NaBu
    )

    # Add Biological/assay noise
    noise = np.random.normal(0, 0.1, size=len(Temp))

    # Titer cannot drop below  g/L in reality
    return np.maximum(0, titer + noise)

In [9]:
# 1. Define the Biological Design Space (The boundaries of our map)
# We tell the algorithm it can only search within these physical limits
space = [
    Real(31.0, 35.0, name='temp'),
    Real(6.8, 7.2, name='ph'),
    Real(3.0, 8.0, name='glucose')
]
space

[Real(low=31.0, high=35.0, prior='uniform', transform='identity'),
 Real(low=6.8, high=7.2, prior='uniform', transform='identity'),
 Real(low=3.0, high=8.0, prior='uniform', transform='identity')]

In [13]:
# 2. Wrap our Biological Ground Truth into an Objective Function
# The decorator maps the 'space' dimensions to the function arguments
@use_named_args(space)
def objective(**params):
    # Retrive the physical parameters
    T = params['temp']
    pH = params['ph']
    G = params['glucose']

    # We lock Feed Start at 4.0 and NaBu at 0.05 (from Chapter 1 insights)
    X_input = np.array([[T, pH, 4.0, G, 0.05]])

    # Run  the experiment (querying our true_bioreactor function from section 1).
    titer = true_bioreactor(X_input)[0]

    # Bayesian Optimization algorithms are built to MINIMIZE by default.
    # To maximize titer, we must return the negative titer.
    return -titer

In [16]:
# 3. Run the GP Optimizer
# n_calls=20 means the algorithm is only allowed to run 20 total "experiments"

print("--- STARTING BAYESIAN OPTIMIZATION ---\n")
res = gp_minimize(
    func=objective,             # The function to minimize
    dimensions=space,           # The biological limits
    acq_func='EI',              # Expected improvement (balances exploration/explotation)
    n_calls=20,                 # Budget: maximum number of bioreactors we can afford
    random_state=5              # Seed for reproducibility
)

# 4. Interpret the Results
print(f'Optimization Complete after {len(res.func_vals)} physical experiments.')
print(f'Maximum Titer Found: {res.fun:.3f} g/L')
print(f'Optimal Parameters Found:')
print(f'    Temperature: {res.x[0]:.2f} °C')
print(f'    pH:          {res.x[1]:.2f}')
print(f'    Glucose:     {res.x[2]:.2f} g/L')

--- STARTING BAYESIAN OPTIMIZATION ---

Optimization Complete after 20 physical experiments.
Maximum Titer Found: -4.266 g/L
Optimal Parameters Found:
    Temperature: 35.00 °C
    pH:          6.80
    Glucose:     5.42 g/L
